In [34]:
from telethon.sync import TelegramClient
import re
import csv
from datetime import datetime
import asyncio

# Конфигурация (заполните своими данными)
api_id = '16494191'
api_hash = 'b450c865395daf03c0853c6b095ab324'
channel_username = 'tondexscreenerupdates'

# Уточненное регулярное выражение для TON-адресов
ton_address_pattern = r'\bEQ[a-zA-Z0-9_-]{43,48}\b'

async def parse_channel():
    async with TelegramClient('session_name', api_id, api_hash) as client:
        channel = await client.get_entity(channel_username)
        
        messages = client.iter_messages(
            channel,
            # offset_date=datetime(2025,12,31),
            # reverse=True,
            # search='новый токен'
        )

        results = []
        cnt = 0
        async for message in messages:
            cnt+=1
            # Проверка наличия текста и даты
            if not message.text or message.date.year not in [2024, 2025]:
                continue
            
            # Приведение к строке и обработка ключевых слов
            text = str(message.text)
            if 'Contract' not in text and 'TON' not in text:
                continue
                
            # Поиск адресов с обработкой ошибок
            # text1 = text[text.find('**contract**:'):]
            # print(text1[text1.find('`')+1:text1.find('\n')-1])
            try:
                addresses = re.findall(ton_address_pattern, text)
            except TypeError:
                continue
                
            # print('yes')
            # break
            if addresses:
                results.append({
                    'date': message.date.strftime("%Y-%m-%d %H:%M:%S"),
                    'addresses': addresses[0],
                    # 'text': message.text[:100] + '...'
                })
        print(len(results), cnt)
        # Сохранение результатов
        with open(f'ton_{channel_username}.csv', 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=['date', 'addresses', 'text'])
            writer.writeheader()
            writer.writerows(results)

# Универсальный запуск
try:
    import nest_asyncio
    nest_asyncio.apply()
    asyncio.run(parse_channel())
except Exception as e:
    print(f"Ошибка выполнения: {str(e)}")


518 576
